# RAG with MongoDb - Data ingestion, Retieval and generation

## Step 1: Data Ingestion and Embedding Generation

### Overview
In this step, we prepare our accommodation data for semantic search by generating vector embeddings and storing them in MongoDB.

### Process

#### 1. Create Embedding Generation Function
We initialize an OpenAI client and define a function to generate semantic vector embeddings using the `text-embedding-3-small` model. This converts text into a 1536-dimensional vector that captures semantic meaning.

#### 2. Parse and Structure Accommodation Data
We load the accommodation halls data from a JSON file and flatten each hall's information into a descriptive text format using the `build_hall_text()` function. This ensures all relevant information (name, type, address, facilities, room details, pricing) is captured in a unified text representation.

#### 3. Generate Embeddings for Each Hall
For each accommodation hall, we:
- Convert the structured text into a semantic vector embedding
- Preserve the original hall metadata (name, type, catering type, etc.)
- Store both the embedding vector and source text in a document structure

#### 4. Connect to MongoDB
We establish a connection to MongoDB using credentials from environment variables and target the `accommodation_embeddings` collection in the `open_day_knowledge` database.

#### 5. Upload Documents to MongoDB
We use MongoDB's bulk write operations with upsert to insert all hall documents with their embeddings. This approach allows us to re-run the ingestion process without creating duplicates.

In [3]:
from openai import OpenAI

# Initialize OpenAI client
openai_client = OpenAI()

#specify the embedding module
model_name = "text-embedding-3-small"

#define the function to generate embedding
def generate_embedding(text, model_name=model_name):
   response = openai_client.embeddings.create(
    input=text,
    model=model_name
   )
   return response.data[0].embedding

In [4]:
len(generate_embedding("This is a sample text to generate embedding."))

1536

In [5]:
## Data ingestion and embedding generation
import json
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the JSON file
json_path = Path("accommodation_halls.json")
with open(json_path, "r", encoding="utf-8") as f:
    halls_data = json.load(f)

def build_hall_text(hall: dict) -> str:
    """Flatten a hall document into a descriptive text string for embedding."""
    parts = []

    if hall.get("name"):
        parts.append(f"Hall name: {hall['name']}")
    if hall.get("type"):
        parts.append(f"Type: {hall['type']}")
    if hall.get("address"):
        parts.append(f"Address: {hall['address']}")
    if hall.get("short_description"):
        parts.append(f"Description: {hall['short_description']}")
    if hall.get("catering_type"):
        parts.append(f"Catering: {hall['catering_type'].replace('_', ' ')}")
    if hall.get("services"):
        parts.append("Services: " + "; ".join(hall["services"]))
    if hall.get("facilities"):
        parts.append("Facilities: " + "; ".join(hall["facilities"]))
    if hall.get("room_types"):
        for room in hall["room_types"]:
            room_text = (
                f"Room type: {room.get('name', '')}, "
                f"{'en-suite' if room.get('ensuite') else 'shared bathroom'}, "
                f"{room.get('occupancy', '')} occupancy, "
                f"{room.get('tenancy_weeks', '')} weeks tenancy"
            )
            if room.get("prices"):
                price = room["prices"][0]
                room_text += (
                    f", £{price.get('per_week_gbp', '')} per week"
                    f" (£{price.get('total_contract_gbp', '')} total)"
                )
            parts.append(room_text)

    return "\n".join(parts)

print(f"Loaded {len(halls_data)} halls from {json_path}")
print("\nSample text for first hall:\n")
print(build_hall_text(halls_data[0]))


Loaded 17 halls from accommodation_halls.json

Sample text for first hall:

Hall name: Butler Court
Type: hall
Address: Butler Court Loughborough University LE11 3TS
Description: Located in East Park, near the School of Design and Creative Arts and our rugby pitches. Butler Court is a very tight knit community where morale is high and there is a great social life. The hall is named after Sir Clifford Butler, Vice-Chancellor of the University from 1975-1985.
Catering: self catered
Services: Fortnightly bedroom cleaning; Weekly cleaning of kitchens and communal areas; Communal bathrooms and toilet facilities will be cleaned a minimum of three times a week
Facilities: Deluxe flats have sofa and TV in the kitchen; Electric hobs in the kitchen; Large games/common room including TV, table tennis and table football; Unlimited wired and wireless internet access; Card/mobile app operated launderette; Bike shed
Room type: Standard, shared bathroom, single occupancy, 41 weeks tenancy, £126.68 per

In [6]:
## Split text chunks and generate embeddings for each hall
documents = []

for hall in halls_data:
    full_text = build_hall_text(hall)

    documents.append({
        "doc_id": hall.get("name", "unknown"),
        "hall_name": hall.get("name"),
        "hall_type": hall.get("type"),
        "catering_type": hall.get("catering_type"),
        "source": "accommodation_halls.json",
        "text": full_text,
        "embedding": generate_embedding(full_text)
    })

print(f"Total halls processed : {len(halls_data)}")
print(f"Total documents created: {len(documents)}")
print(f"\nSample document (minus embedding):")
sample = {k: v for k, v in documents[0].items() if k != "embedding"}
print(sample)
print(f"\nEmbedding vector length: {len(documents[0]['embedding'])}")



Total halls processed : 17
Total documents created: 17

Sample document (minus embedding):
{'doc_id': 'Butler Court', 'hall_name': 'Butler Court', 'hall_type': 'hall', 'catering_type': 'self_catered', 'source': 'accommodation_halls.json', 'text': 'Hall name: Butler Court\nType: hall\nAddress: Butler Court Loughborough University LE11 3TS\nDescription: Located in East Park, near the School of Design and Creative Arts and our rugby pitches. Butler Court is a very tight knit community where morale is high and there is a great social life. The hall is named after Sir Clifford Butler, Vice-Chancellor of the University from 1975-1985.\nCatering: self catered\nServices: Fortnightly bedroom cleaning; Weekly cleaning of kitchens and communal areas; Communal bathrooms and toilet facilities will be cleaned a minimum of three times a week\nFacilities: Deluxe flats have sofa and TV in the kitchen; Electric hobs in the kitchen; Large games/common room including TV, table tennis and table football; U

In [18]:
## Connect to MongoDB
import os
from dotenv import load_dotenv
from pymongo import MongoClient

# Load environment variables from .env file
# override=True forces a re-read even if variables are already set in this session
load_dotenv(override=True)

MONGODB_URI = os.getenv("MONGODB_URI")
mongodb_client = MongoClient(MONGODB_URI)

# Send a ping to confirm a successful connection
try:
    mongodb_client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

db = mongodb_client["open_day_knowledge"]
collection = db["accommodation_embeddings"]

print(f"Connected to database : {db.name}")
print(f"Target collection    : {collection.name}")


Pinged your deployment. You successfully connected to MongoDB!
Connected to database : open_day_knowledge
Target collection    : accommodation_embeddings


In [8]:
## Insert documents with embeddings into MongoDB
# Uses replace_one with upsert so re-running won't create duplicate hall documents

from pymongo import ReplaceOne

if documents:
    ops = [
        ReplaceOne({"doc_id": doc["doc_id"]}, doc, upsert=True)
        for doc in documents
    ]
    result = collection.bulk_write(ops, ordered=False)
    print(f"Upserted {result.upserted_count} new | Modified {result.modified_count} existing documents in '{collection.name}'")
else:
    print("No documents to insert.")


Upserted 17 new | Modified 0 existing documents in 'accommodation_embeddings'


## Step 2: Vector Search Index Creation and Query Retrieval

### Overview
In this step, we create a vector search index in MongoDB and implement the retrieval mechanism for our RAG system. This enables semantic similarity search across our accommodation embeddings.

### Process

#### 1. Create Vector Search Index
We define a `vectorSearch` index on the `embedding` field with the following specifications:
- **Field Type**: vector
- **Dimensions**: 1536 (matching our embedding model output)
- **Path**: embedding field in documents
- **Similarity Metric**: cosine (measures semantic similarity between vectors)

This index allows MongoDB to efficiently perform approximate nearest neighbor searches on high-dimensional embedding vectors.

#### 2. Index Readiness Verification
After creating the search index, we poll MongoDB to confirm the index is queryable. This ensures the index has completed its initial synchronization before we begin querying.

#### 3. Query Embedding Generation
For each user query, we:
- Generate a vector embedding using the same `text-embedding-3-small` model used for document embeddings
- This ensures queries and documents are represented in the same semantic space for accurate comparison

#### 4. Vector Search Query Execution
We construct an aggregation pipeline with a `$vectorSearch` stage that:
- Searches against the `vector_index`
- Compares the query embedding against all document embeddings using cosine similarity
- Returns the top matching accommodation halls based on semantic relevance
- Projects only the `text` field containing the full hall description

#### 5. Query Results Function
The `get_query_results()` function encapsulates the retrieval process, executing the vector search and returning an array of the most relevant accommodation documents from MongoDB. This retrieved context is then passed to the language model for answer generation.

In [9]:
## query with a search index
from pymongo.operations import SearchIndexModel
import time

# Create your index model, then create the search index
index_name="vector_index"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 1536,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)

result = collection.create_search_index(model=search_index_model)
print("New search index named " + result + " is building.")


New search index named vector_index is building.


In [10]:
# Wait for initial sync to complete
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None

if predicate is None:
  predicate = lambda index: index.get("queryable") is True
while True:
  indices = list(collection.list_search_indexes(result))
  if len(indices) and predicate(indices[0]):
    break
  time.sleep(5)
print(result + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


## Testing Vector Search Index

Verify that the vector search index is functioning correctly before implementing the full retrieval pipeline.

In [11]:
# test that the search index works for this database
query_embedding = generate_embedding("What accommodation halls have en-suite rooms?")
result = collection.aggregate([
  {
    "$vectorSearch": {
      "index": "vector_index",
      "path": "embedding",
      "queryVector":query_embedding,
      "numCandidates":1536,
      "limit": 5
    }
  }
])

In [12]:
array_of_results = []
for doc in result:
    array_of_results.append(doc)

In [13]:
array_of_results

[{'_id': ObjectId('69a708cba10917389b716f67'),
  'text': 'Hall name: Royce\nType: hall\nAddress: Royce Hall Loughborough University LE11 3TJ\nDescription: Located in the popular student village, Royce consists of 19 accommodation blocks of two and three storeys. It is close to lecture halls, sports facilities, the library and is only a 20-minute walk from town. The hall is named after Sir Henry Royce, founder member of Rolls-Royce Ltd, a pioneer of aeronautical engineering in education.\nCatering: catered\nServices: Fortnightly bedroom cleaning; Weekly cleaning of kitchens and communal areas; Communal bathrooms and toilet facilities will be cleaned a minimum of three times a week\nFacilities: 2 - 8 students share a kitchenette (number sharing facilities is higher in some larger blocks); Large common/games room - Freeview TV, table football, darts and more (facilities subject to change); The University network and internet access are available by both wired and wireless networks; Card/m

In [14]:
# Define a function to run vector search queries
"""Gets results from a vector search query."""
def get_query_results(query):
    query_embedding = generate_embedding(query)
    print(query_embedding)
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates":300,
                "limit": 20
            }
        }, {
            "$project": {
                "_id": 0,
                "text": 1
            }
        }
    ]

    results = collection.aggregate(pipeline)
    print(results)

    array_of_results = []
    for doc in results:
        array_of_results.append(doc)
    return array_of_results



In [15]:
get_query_results("how much is butler court accomodation?")

[-0.03183870017528534, -0.026496365666389465, 0.04408435523509979, -0.030519938096404076, -0.07594996690750122, -0.019996749237179756, -0.0033507340122014284, -0.009083314798772335, 0.02165193110704422, -0.0427117645740509, 0.001983190421015024, -0.0442996621131897, 0.013591330498456955, -0.003673696191981435, -0.015932807698845863, 0.006745202466845512, -0.013638429343700409, 0.03848634287714958, -0.024760443717241287, 0.063408263027668, -0.042254235595464706, 0.012541702948510647, -0.01094034779816866, -0.03294215351343155, -0.0040773991495370865, -0.03716757893562317, 0.0034516595769673586, -0.09161364287137985, 0.07320478558540344, 0.022311313077807426, 0.01120948325842619, -0.03878239169716835, 0.04354608431458473, -0.028339942917227745, -0.0420389249920845, 0.03730214759707451, -0.016484534367918968, 0.028716731816530228, 0.00016410942771472037, -0.04042411595582962, -0.017251569777727127, 0.03442239761352539, -0.035310544073581696, -0.012043802998960018, -0.018381938338279724, 0

[{'text': 'Hall name: Butler Court\nType: hall\nAddress: Butler Court Loughborough University LE11 3TS\nDescription: Located in East Park, near the School of Design and Creative Arts and our rugby pitches. Butler Court is a very tight knit community where morale is high and there is a great social life. The hall is named after Sir Clifford Butler, Vice-Chancellor of the University from 1975-1985.\nCatering: self catered\nServices: Fortnightly bedroom cleaning; Weekly cleaning of kitchens and communal areas; Communal bathrooms and toilet facilities will be cleaned a minimum of three times a week\nFacilities: Deluxe flats have sofa and TV in the kitchen; Electric hobs in the kitchen; Large games/common room including TV, table tennis and table football; Unlimited wired and wireless internet access; Card/mobile app operated launderette; Bike shed\n\nRoom type: Shared Twin with En-Suite\nEn-suite: Yes\nBathroom: en-suite\nOccupancy: double\nTenancy: 41 weeks\nAvailable to: undergraduate, i

# Step 3: Answer Generation with Retrieved Context

## Overview
In this step, we implement the generation phase of our RAG system. We combine the retrieved accommodation data with a language model to generate accurate, context-aware answers to user queries.

## Process

### 1. Answer Generation Function
We define an `answer_query()` function that orchestrates the complete RAG pipeline:
- Takes a user query and model name as inputs
- Retrieves relevant accommodation documents using the vector search from the previous step
- Constructs a prompt that includes the retrieved context and user question
- Sends the prompt to the language model (GPT-4o) for generation

### 2. Context Assembly
The function merges all retrieved document texts into a single context string, providing the language model with comprehensive accommodation information relevant to the user's query.

### 3. Prompt Engineering
We craft a system prompt that:
- Instructs the model to act as a helpful university accommodation assistant
- Constrains responses to use ONLY the provided context
- Ensures specific information (hall names, prices, features) is always included
- Handles out-of-context queries gracefully by indicating missing information

### 4. Language Model Integration
We use OpenAI's Chat Completion API (GPT-4o model) to generate responses based on the contextualized prompt, ensuring accurate and relevant answers grounded in the actual accommodation data.

### 5. Testing the RAG System
We execute a series of test questions covering common user inquiries about accommodation halls. These tests validate that the system correctly retrieves relevant data and generates appropriate responses across different query types.

In [16]:
## answer the question with the retrieved results using a language model
def answer_query(query,model_name):
    
  context_docs = get_query_results(query)
  context_string = " ".join([doc["text"] for doc in context_docs])

  # Construct prompt for the LLM using the retrieved documents as the context
  prompt =  f"""You are a helpful university accommodation assistant.
  Use ONLY the context below to answer the question.
  If the answer is not in the context, say you don't have that information.
  Be specific: always name the hall when giving prices or features.

  CONTEXT:
  {context_string}

  QUESTION: {query}
  """

  completion = openai_client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": prompt}]
    )
  
  print(f"Q: {query}\n")
  print(completion.choices[0].message.content)
  print("\n" + "="*60 + "\n")

In [17]:
# list of questions abou taccomadation halls to test the RAG chatbot
questions = [
    "how much is butler court accomodation?",
    "which halls have en-suite rooms?",
    "which halls have catering included?",
    "what facilities does piper court have?",
    "which hall has the cheapest rooms?"
]

# OpenAI model to use
model_name = "gpt-4o"

for question in questions:
    answer_query(question, model_name)

[-0.031769197434186935, -0.026451896876096725, 0.04409995675086975, -0.030476892367005348, -0.07597684860229492, -0.020003825426101685, -0.0034040831960737705, -0.009032683447003365, 0.021673055365681648, -0.04278072714805603, 0.001980526838451624, -0.04434226453304291, 0.013589409179985523, -0.003674996318295598, -0.015951907262206078, 0.006757685448974371, -0.013643255457282066, 0.03849996253848076, -0.0247153602540493, 0.06337685883045197, -0.04229611158370972, 0.012546141631901264, -0.010950950905680656, -0.03298073634505272, -0.004011534620076418, -0.03709996119141579, 0.003422592766582966, -0.09153836965560913, 0.07323069870471954, 0.02233267016708851, 0.011159604415297508, -0.0387691929936409, 0.0435076467692852, -0.02832304872572422, -0.04205380380153656, 0.03731534630060196, -0.01649036817252636, 0.028699971735477448, 0.0001490232825744897, -0.040384575724601746, -0.017230750992894173, 0.03435381129384041, -0.03529611974954605, -0.012007679790258408, -0.018374981358647346, 0.0